In [ ]:
import requests
from bs4 import BeautifulSoup
import textwrap
import time

In [ ]:
# Function to fetch HTML content from a URL with retry mechanism
def fetch_html(url, retries=3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    attempt = 0
    while attempt < retries:
        try:
            response = requests.get(url, headers=headers, timeout=20)  # Increased timeout and added User-Agent
            response.raise_for_status()  # Check for request errors
            return response.content
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            attempt += 1
            if attempt < retries:
                print(f"Retrying {url} ({attempt}/{retries})...")
                time.sleep(2)  # Wait before retrying
            else:
                print(f"Failed to fetch {url} after {retries} attempts.")
                return None

# Function to remove header, footer, and extract the remaining content
def extract_main_content(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    # Remove header and footer
    if soup.header:
        soup.header.decompose()
    if soup.footer:
        soup.footer.decompose()

    # Remove page-head element by class
    page_head = soup.find(class_="page-head")
    if page_head:
        page_head.decompose()

    # Remove specific elements by their classes
    specific_classes = [
        "bUqfOz", "hEiKeJ", "gySqrp", "customHeader", "headernavbar", "breadcrumb",
        "customfooter", "content_top", "itr-season-banner", "header-wrapper",
        "common-wrapper", "common-right", "common-left", "container-fluid row",
        "bread_crumbs", "topic", "top-header", "navbar", "nav clearfix",
        "row breadcrumb-outer", "steps px-0", "menu_wrapper", "region region-user-menu",
        "myheadbtnhdr", "top-bar","profile-download","easy-breadcrumb","min_storng"
    ]

    for class_name in specific_classes:
        elements = soup.find_all(class_=class_name)
        for element in elements:
            element.decompose()

    # Get remaining text from the body
    body_content = soup.body.get_text(separator='\n').strip() if soup.body else ""
    return body_content

# Function to chunk text with overlap
def chunk_text(text, chunk_size=100, overlap=35):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
    return chunks

# List of URLs with meaningful names
urls_with_names = {
  "https://www.presidentofindia.gov.in/Profile": "President Profile",
"https://vicepresidentofindia.nic.in/profile" : "Vice President Profile",
"https://www.pmindia.gov.in/en/pms-profile/" : "Prime Minister Profile",
}

# Initialize an empty list for chunks
Profile = []

for url, name in urls_with_names.items():
    try:
        html_content = fetch_html(url)
        if html_content:  # Proceed only if HTML content is successfully fetched
            print(f"Fetched HTML for URL: {url}")  # Debug print

            extracted_text = extract_main_content(html_content)
            if extracted_text:
                print(f"Extracted content from {url} using extract_main_content.")  # Debug print
            else:
                print(f"Can't scrape content from {url} using the <h> and <p> tags")  # Debug print
                soup = BeautifulSoup(html_content, 'html.parser')
                data = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
                scrap_data = [d.text.strip() for d in data if d.text.strip()]  # Strip and check for empty text
                extracted_text = '\n'.join(scrap_data)

            # Only chunk if there's actual text
            if extracted_text.strip():
                chunked_text = chunk_text(extracted_text, chunk_size=180, overlap=45)
                for i, chunk in enumerate(chunked_text):
                    # Wrap text to ensure it fits within the desired width
                    wrapped_text = textwrap.fill(chunk, width=80)

                    Profile.append(f"{name}:\n{wrapped_text}")
            else:
                print(f"No valid content extracted from {url}.")
        else:
            print(f"Failed to fetch content from {url}")  # Debug print

    except Exception as e:
        print(f"Error fetching or processing {url}: {e}")

    time.sleep(1)




Fetched HTML for URL: https://www.presidentofindia.gov.in/Profile
Extracted content from https://www.presidentofindia.gov.in/Profile using extract_main_content.
Fetched HTML for URL: https://vicepresidentofindia.nic.in/profile
Extracted content from https://vicepresidentofindia.nic.in/profile using extract_main_content.
Fetched HTML for URL: https://www.pmindia.gov.in/en/pms-profile/
Extracted content from https://www.pmindia.gov.in/en/pms-profile/ using extract_main_content.


In [ ]:
for chunk in Profile:
    print(chunk)
    print("-" * 80)

President Profile:
Skip to main content Profile Profile of the President Smt. Droupadi Murmu was
sworn in as the 15th President of India on 25 July, 2022. Previously, she was
the Governor of Jharkhand from 2015 to 2021. She has devoted her life to
empowering the downtrodden and the marginalised sections and deepening the
democratic values. Early Life and Education Born in a Santhali tribal family on
20 June, 1958 at Uparbeda village, Mayurbhanj, Odisha, Smt. Murmu’s early life
was marked by hardships and struggle. On completion of primary education from
the village school, she went to Bhubaneswar on her own initiative to continue
her studies. She earned the degree of Bachelor of Arts from Ramadevi Women’s
College, Bhubaneswar and became the first woman from her village to receive
college education. Professional Career From 1979 to 1983, Smt. Murmu served as a
Junior Assistant in the Irrigation and Power Department, Government of Odisha.
Later, she served as an honorary teacher at Sri A